# Focus on the Unitary Dilation

In [1]:
using LinearAlgebra
using Plots
using Revise
using Latexify

includet("../julia/src/UnitaryDilation/UnitaryDilation.jl")
includet("../julia/src/PetzMaps.jl")
includet("../julia/src/utils.jl")

using .UnitaryDilation
using .PetzMaps

In [2]:
function get_kraus_operators(noise, gamma, t)
  if noise == "amplitude_damping"
    return get_amplitudedamping_operators(gamma, t)
  elseif noise == "dephasing"
    return get_dephasing_operators(gamma, t)
  elseif noise == "bitflip"
    return get_bitflip_operators(gamma, t)
  else
    error("Unknown noise model: $noise")
  end

end

function apply_noise(model, ρ, n_qubits)
  ρf = apply_channel(model.kraus_fwd, ρ, n_qubits)
  # Enforce physicality (hermitianicity and trace 1)
  # enforce_physical!(ρf)
  return ρf
end


function recovery(model, ρ)
  ρr, η = apply_petz_collision(model, ρ)
  # enforce_physical!(ρr)
  return ρr, η
end

recovery (generic function with 1 method)

Initialize the system

In [3]:
n_qubits = 2
beta = 1.0
gamma = 1.0
dt = 0.5
noise = "amplitude_damping"
n_steps = 2

# Take the relevant Kraus operators
kraus_single_qubit = get_kraus_operators(noise, gamma, dt)

# Choose a reference state for the recovery
sigma = thermal_state(n_qubits, beta)
# The initial state is the reference state itself
ρ0 = sigma

latexify(sigma; fmt="%.3f")  # display as LaTeX table

L"\begin{equation}
\left[
\begin{array}{cccc}
0.440+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
-0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
-0.0\mathit{i} & -0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} \\
-0.0\mathit{i} & -0.0\mathit{i} & -0.0\mathit{i} & 0.440+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

### Step 1
We apply the noise and recover the state, both with the collision model and the standard Petz map.
- `rho1`: control state where only noise is applied,
- `rho2`: state where noise + collision is applied,
- `rho3`: state where noise + Petz map is applied

In [4]:
rho1 = copy(ρ0)
rho2 = copy(ρ0)
rho3 = copy(ρ0)

rho1 = apply_channel(kraus_single_qubit, rho1, n_qubits)
rho2 = apply_channel(kraus_single_qubit, rho2, n_qubits)
rho3 = apply_channel(kraus_single_qubit, rho3, n_qubits)

# This will be our reference, non-recovered state
display(latexify(rho1; fmt="%.3f"))
println("Fidelity wrt initial state: ", fidelity(ρ0, rho1))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.555+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.162+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

Fidelity wrt initial state: 0.8934562192902762


### Step 2
We recover the state `rho2` with the classic Petz map defined by the Kraus operators of the selected noise.
Remember that `rho2` starts in the same state of `rho1` after the noise.

In [5]:
rho2 = recovery_map(kraus_single_qubit, sigma, rho2, n_qubits)

display(latexify(rho2; fmt="%.3f"))
println("Fidelity wrt initial state: ", fidelity(ρ0, rho2))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.440+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.440+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

Fidelity wrt initial state: 1.0


### Step 3.1: Check Partial Trace

In [6]:
display(latexify(rho1; fmt="%.3f"))
rho3 = copy(rho1)

d_a = length(kraus_single_qubit)

η = rand(ComplexF64, d_a, d_a)
η = (η + η') / 2  # make it Hermitian
η = η / tr(η)  # normalize to have trace 1
display(latexify(η; fmt="%.3f"))

rho_total = kron(rho3, η)

# 2. Apply Unitary: U (ρ ⊗ |0><0|) U†
U = I
rho_after = U * rho_total * U'

rho3, η = partial_traces(rho_after, 2^n_qubits, d_a)
display("---")
display(latexify(rho3; fmt="%.3f"))
display(latexify(η; fmt="%.3f"))
if isapprox(rho3, rho1)
  println("The partial trace works.")
else
  println("ERROR IN THE PARTIAL TRACE!")
end

L"\begin{equation}
\left[
\begin{array}{cccc}
0.555+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.162+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.778+0.000\mathit{i} & 0.631+0.021\mathit{i} \\
0.631-0.021\mathit{i} & 0.222+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

"---"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.555+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.141+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.162+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cc}
0.778+0.000\mathit{i} & 0.631+0.021\mathit{i} \\
0.631-0.021\mathit{i} & 0.222+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

The partial trace works.


### Step 3.2: Check Backward Kraus Operators
These are the Kraus Operators of the Petz Map:
$$
R_i = \sigma^{\frac{1}{2}}K_i^\dagger\left(\sum_j K_j\sigma K_j\right)^{-\frac{1}{2}}
$$

In [7]:
function kraus_backward(kraus_fwd, sigma)
    sigma_out = sum(K * sigma * K' for K in kraus_fwd)
    sigma_out_inv_sqrt = inv(sqrt(sigma_out))
    sigma_sqrt = sqrt(Hermitian(sigma))
    kraus_rec = [sigma_sqrt * K' * sigma_out_inv_sqrt for K in kraus_fwd]
    return kraus_rec
end

model = CollisionModel(kraus_single_qubit, sigma, n=n_qubits)
kraus_rec = model.kraus_rec

rho3 = copy(rho1)

# kraus_rec = kraus_backward(kraus_single_qubit, sigma)

rho3 = apply_channel(kraus_rec, rho3)
display(latexify(rho3; fmt="%.10f"))
println("Fidelity wrt initial state: ", fidelity(ρ0, rho3))

L"\begin{equation}
\left[
\begin{array}{cccc}
0.4403985389+0.0000000000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0596014610+0.0000000000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0596014610+0.0000000000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.4403985387+0.0000000000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

Fidelity wrt initial state: 0.9999999995999997


### Step 3.3: Check Unitary Dilation
We check this using the standard forward operators of known noises

In [8]:
function unitary_dilation(K_ops, d_s, d_a)
    """
    Apply a unitary dilation to system+ancilla that implements Kraus operators on the system.
    
    Arguments:
    - K_ops: vector of Kraus operators [K_0, K_1, ..., K_{n-1}]
    - d_s: dimension of system
    - d_a: dimension of ancilla (must be ≥ length(K_ops))
    
    The unitary acts as: U = Σ_k K_k ⊗ |k⟩⟨0|
    assuming the ancilla starts in |0⟩ state.
    
    Returns:
    - U: the dilation unitary matrix
    """
    
    n_kraus = length(K_ops)
    
    if d_a < n_kraus
        error("Ancilla dimension ($d_a) must be ≥ number of Kraus operators ($n_kraus)")
    end
    
    d_total = d_s * d_a
    U = zeros(ComplexF64, d_total, d_total)
    
    # Build unitary dilation: U|j⟩_s|0⟩_a = Σ_k K_k|j⟩_s ⊗ |k⟩_a
    # For kron(system, ancilla), index = (i-1)*d_a + k
    
    for i in 1:d_s
        for j in 1:d_s
            # Input state: |j⟩_s|0⟩_a
            idx_in = (j-1)*d_a + 1
            
            # Apply each Kraus operator
            for (k_idx, K) in enumerate(K_ops)
                # Output: K_k|j⟩_s ⊗ |k-1⟩_a (k_idx starts at 1, so ancilla state is k_idx-1)
                ancilla_state = k_idx  # This is the ancilla index (1-indexed)
                idx_out = (i-1)*d_a + ancilla_state
                U[idx_out, idx_in] = K[i, j]
            end
        end
    end
    
    # For ancilla states |k⟩ with k > 0, we need to complete the unitary
    # Apply identity on these subspaces
    for k in 2:d_a
        for i in 1:d_s
            idx = (i-1)*d_a + k
            if U[idx, idx] == 0.0  # Only if not already filled
                U[idx, idx] = 1.0
            end
        end
    end
    
    return U
end

# Define dimensions
d_s = 2^n_qubits  # 2-qubit system
d_a = 4  # 1-qubit ancilla

# Start from the noisy state
rho3 = copy(rho1)

η = zeros(ComplexF64, d_a, d_a)
η[1, 1] = 1.0  # ancilla in |0⟩ state

ρ_total = kron(rho3, η)

# Apply unitary dilation
U = unitary_dilation(kraus_rec, d_s, d_a)
ρ_new = U * ρ_total * U'


# Verify: trace out ancilla and compare
ρ_system_new, _ = partial_traces(ρ_new, d_s, d_a)
ρ_system_expected = apply_channel(kraus_rec, rho3)  # Direct Kraus application

println("Difference: ", norm(ρ_system_new - ρ_system_expected))
display(latexify(ρ_system_new; fmt="%.3f"))
display(latexify(ρ_system_expected; fmt="%.3f"))

Difference: 0.0


L"\begin{equation}
\left[
\begin{array}{cccc}
0.440+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.440+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

L"\begin{equation}
\left[
\begin{array}{cccc}
0.440+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.060+0.000\mathit{i} & 0.0\mathit{i} \\
0.0\mathit{i} & 0.0\mathit{i} & 0.0\mathit{i} & 0.440+0.000\mathit{i} \\
\end{array}
\right]
\end{equation}
"

***It works!***

---

## Recover random states
This is, on average, better than noise, only if $\beta$ and $n$ are big enough.

In [28]:
n_qubits = 3
beta = 1.0
gamma = 1.0
dt = 0.1
noise = "amplitude_damping"
n_states = 50

states = []
noise_fidelities = []
recovery_fidelities = []
collision_fidelities = []

# Take the relevant Kraus operators
kraus_single_qubit = get_kraus_operators(noise, gamma, dt)

# Choose a reference state for the recovery
sigma = thermal_state(n_qubits, beta)

# set_description(outer_bar, "$n/$(round(beta, digits=2))")
ψ0 = zeros(ComplexF64, 2^n_qubits)

f1s = []
f2s = []
f3s = []

for _ in 1:n_states
    ψ0 = random_state(n_qubits)
    ρ0 = ψ0 * ψ0'
    ρ1 = deepcopy(ρ0)
    ρ2 = deepcopy(ρ0)
    ρ3 = deepcopy(ρ0)

    # Evolve the states
    kraus = get_kraus_operators(noise, gamma, dt)
    collision_model = CollisionModel(kraus, sigma, n=n_qubits)

    ρ1 = apply_channel(kraus, ρ1, n_qubits)
    ρ2 = apply_channel(kraus, ρ2, n_qubits)
    ρ3 = apply_channel(collision_model.kraus_fwd, ρ3)
    # Recover only the second state and third state (with the collision)
    ρ2 = recovery_map(kraus, sigma, ρ2, n_qubits)
    ρ3, η = apply_collision(collision_model, ρ3)

    # Compute fidelity
    f1 = fidelity(ρ0, ρ1)
    f2 = fidelity(ρ0, ρ2)
    f3 = fidelity(ρ0, ρ3)
    append!(f1s, f1)
    append!(f2s, f2)
    append!(f3s, f3)
    push!(states, ψ0)
end

println(sum(f1s) / n_states)
println(sum(f2s) / n_states)
println(sum(f3s) / n_states)

0.872875348671068
0.9321520437011708
0.9321520433373189


In [29]:
for i in 1:n_states
    println("$i) $(f1s[i]) \t $(f2s[i]) \t $(f3s[i])")
end

1) 0.9718650595440261 	 0.9682624811092053 	 0.9682624808950065
2) 0.969395876062314 	 0.9662552209700128 	 0.966255220754791
3) 0.858131374409417 	 0.9124784368147024 	 0.9124784364635782
4) 0.8781818731227388 	 0.916604551834173 	 0.9166045515211715
5) 0.8123940307369955 	 0.9130337188373151 	 0.9130337183748142
6) 0.7562849055229205 	 0.9340336329490756 	 0.9340336323001656
7) 0.9266152574251053 	 0.9368894608579242 	 0.9368894606117336
8) 0.9400483195374832 	 0.9449904098830592 	 0.9449904096493364
9) 0.8981830951595308 	 0.9232504039187123 	 0.923250403637541
10) 0.8229496197222357 	 0.9116364714299986 	 0.9116364709963112
11) 0.8339801286482398 	 0.9110013153441656 	 0.9110013149385672
12) 0.8507325036589051 	 0.9116141280007566 	 0.9116141276339528
13) 0.9634814114913902 	 0.9615832698485018 	 0.9615832696304983
14) 0.9870466158666155 	 0.9813307589331799 	 0.981330758723486
15) 0.9138772469202769 	 0.9301818756105801 	 0.9301818753501807
16) 0.9401870835047412 	 0.9450795222518